In [2]:
import torch

In [3]:
# model_path = '/home/users/bradlesc/projects/ClimSim/models/models/saved_model/v2rh_unet_nonaggressive_cliprh_mae/model.pt'
model_path = "/home/users/bradlesc/projects/ClimSim/models/models/saved_model/v5_unet_nonaggressive_cliprh_mae/model.pt"
model = torch.load(model_path, map_location="cpu", weights_only=False)


/home/users/bradlesc/projects/ClimSim/.venv/lib/python3.12/site-packages/torch/serialization.py:1493: UserWarning: 'torch.load' received a zip file that looks like a TorchScript archive dispatching to 'torch.jit.load' (call 'torch.jit.load' directly to silence this warning)
  warnings.warn(


In [4]:
if isinstance(model, dict) and 'state_dict' in model:
    state_dict = model['state_dict']
else:
    state_dict = model.state_dict()

# print('Model keys:', list(state_dict.keys())[:10])  # print first 10 keys

In [94]:
for name, module in model.named_modules():
    if "32_down" in name:
        print(name, "->", module)

enc.32_down -> RecursiveScriptModule(
  original_name=UNetBlock_noatten
  (norm0): RecursiveScriptModule(original_name=GroupNorm)
  (conv0): RecursiveScriptModule(original_name=Conv1d)
  (norm1): RecursiveScriptModule(original_name=GroupNorm)
  (conv1): RecursiveScriptModule(original_name=Conv1d)
  (skip): RecursiveScriptModule(original_name=Conv1d)
)
enc.32_down.norm0 -> RecursiveScriptModule(original_name=GroupNorm)
enc.32_down.conv0 -> RecursiveScriptModule(original_name=Conv1d)
enc.32_down.norm1 -> RecursiveScriptModule(original_name=GroupNorm)
enc.32_down.conv1 -> RecursiveScriptModule(original_name=Conv1d)
enc.32_down.skip -> RecursiveScriptModule(original_name=Conv1d)


In [97]:
down_block = dict(model.named_modules())["enc.32_down"]

for name, m in down_block.named_modules():
    if isinstance(m, torch.nn.Conv1d):
        print(f"\n{name}")
        print(f"  in_channels:  {m.in_channels}")
        print(f"  out_channels: {m.out_channels}")
        print(f"  kernel_size:  {m.kernel_size}")
        print(f"  stride:       {m.stride}")
        print(f"  padding:      {m.padding}")
    else:
        print(f"{name}: {m}")

: RecursiveScriptModule(
  original_name=UNetBlock_noatten
  (norm0): RecursiveScriptModule(original_name=GroupNorm)
  (conv0): RecursiveScriptModule(original_name=Conv1d)
  (norm1): RecursiveScriptModule(original_name=GroupNorm)
  (conv1): RecursiveScriptModule(original_name=Conv1d)
  (skip): RecursiveScriptModule(original_name=Conv1d)
)
norm0: RecursiveScriptModule(original_name=GroupNorm)
conv0: RecursiveScriptModule(original_name=Conv1d)
norm1: RecursiveScriptModule(original_name=GroupNorm)
conv1: RecursiveScriptModule(original_name=Conv1d)
skip: RecursiveScriptModule(original_name=Conv1d)


In [ ]:
# skip_conv = down_block.skip
# print("Skip conv parameters:")
# print(skip_conv.in_channels, skip_conv.out_channels, skip_conv.stride, skip_conv.kernel_size)


Skip conv parameters:


AttributeError: 'RecursiveScriptModule' object has no attribute 'stride'

In [99]:
def inspect_block(model, block_name):
    """
    Prints all Conv1d-like parameters (including scripted ones) in a given block.
    Works for normal nn.Module or TorchScript RecursiveScriptModule.
    """
    # Locate the submodule
    parts = block_name.split(".")
    m = model
    for p in parts:
        if hasattr(m, p):
            m = getattr(m, p)
        elif p in getattr(m, "_modules", {}):
            m = m._modules[p]
        else:
            raise ValueError(f"Cannot find submodule '{block_name}'")
    
    print(f"\nInspecting block: {block_name}\n{'='*50}")
    
    for name, sub in m._modules.items():
        # Try to print Conv1d attributes
        if hasattr(sub, "weight") and hasattr(sub, "bias"):
            print(f"Submodule: {name}")
            # read Conv1d attributes safely
            for attr in ["in_channels", "out_channels", "kernel_size", "stride", "padding"]:
                val = getattr(sub, attr, None)
                if val is not None:
                    print(f"  {attr}: {val}")
            # if scripted Conv1d, attributes might be stored as buffers
            if not hasattr(sub, "stride") and hasattr(sub, "_c"):
                try:
                    print("  (TorchScript conv attributes)")
                    print("   ", sub._c.get_attributes())
                except Exception:
                    pass
            print()
        else:
            # Recursively look deeper
            if len(sub._modules) > 0:
                for inner_name, inner in sub._modules.items():
                    if hasattr(inner, "weight"):
                        print(f"Submodule: {name}.{inner_name}")
                        for attr in ["in_channels", "out_channels", "kernel_size", "stride", "padding"]:
                            val = getattr(inner, attr, None)
                            if val is not None:
                                print(f"  {attr}: {val}")
                        print()

In [100]:
inspect_block(model, "enc.32_down")



Inspecting block: enc.32_down
Submodule: norm0
  (TorchScript conv attributes)

Submodule: conv0
  in_channels: 128
  out_channels: 128
  (TorchScript conv attributes)

Submodule: norm1
  (TorchScript conv attributes)

Submodule: conv1
  in_channels: 128
  out_channels: 128
  (TorchScript conv attributes)

Submodule: skip
  in_channels: 128
  out_channels: 128
  (TorchScript conv attributes)



In [6]:
import math

def get_submodule_by_name(model, name):
    """Locate nested submodule by dotted name (works for scripted modules too)."""
    parts = name.split(".")
    m = model
    for p in parts:
        # try attribute, then _modules, then getattr fallback
        if hasattr(m, p):
            m = getattr(m, p)
        elif isinstance(getattr(m, "_modules", None), dict) and p in m._modules:
            m = m._modules[p]
        else:
            raise KeyError(f"Cannot find submodule '{name}' (stuck at '{p}')")
    return m

def infer_conv_attributes(conv_module, test_length=64):
    """
    For a Conv-like module (scripted or regular), run a small probe to infer:
      - weight shape -> kernel_size
      - bias existence
      - output length for input length = test_length -> infer stride
      - infer padding from conv formula
    Returns dict of inferred attributes.
    """
    info = {}
    # Try to access weight tensor
    weight = None
    bias = None
    try:
        weight = conv_module.weight
    except Exception:
        # scripted modules sometimes expose parameters in ._parameters or via attributes
        try:
            # try _parameters dict
            weight = dict(conv_module._parameters).get("weight", None)
        except Exception:
            weight = None

    try:
        bias = conv_module.bias
    except Exception:
        try:
            bias = dict(conv_module._parameters).get("bias", None)
        except Exception:
            bias = None

    if weight is None:
        info['weight_shape'] = None
    else:
        # weight shape: (out_channels, in_channels, kernel_size)
        ws = tuple(weight.shape)
        info['weight_shape'] = ws
        # kernel size may be tuple or int
        if len(ws) >= 3:
            info['kernel_size'] = ws[2]
            info['out_channels'] = ws[0]
            info['in_channels'] = ws[1]

    info['has_bias'] = (bias is not None)

    # Now run a forward probe if callable to get output length
    # Create a dummy tensor with batch=1, channels=in_channels, length=test_length
    if info.get('in_channels') is None:
        info['probe_success'] = False
        return info

    B = 1
    C = int(info['in_channels'])
    L_in = int(test_length)
    x = torch.randn(B, C, L_in)

    try:
        # Some scripted submodules require using .forward or are directly callable.
        # We'll try both.
        try:
            y = conv_module(x)
        except Exception:
            y = conv_module.forward(x)
    except Exception as e:
        info['probe_success'] = False
        info['probe_error'] = repr(e)
        return info

    info['probe_success'] = True
    info['L_in'] = L_in
    info['L_out'] = y.shape[-1]

    # Infer stride as integer ratio if possible (L_in / L_out)
    # Acceptable strides are integer (1 or 2) typically for this model.
    ratio = L_in / info['L_out'] if info['L_out'] != 0 else None
    if ratio is not None and abs(round(ratio) - ratio) < 1e-6:
        s = int(round(ratio))
    else:
        # fallback: compute theoretical s candidates 1 or 2 by checking formula solvability
        s = None
        for candidate in (1, 2, 3, 4):
            # compute required padding p from conv formula:
            k = info.get('kernel_size', None)
            if k is None: 
                continue
            # L_out = floor((L_in + 2p - (k-1) -1)/s + 1)
            # ignore floor by trying integer p in small range
            found = False
            for p in range(0, 5):
                L_calc = math.floor((L_in + 2*p - (k-1) - 1) / candidate + 1)
                if L_calc == info['L_out']:
                    s = candidate
                    found = True
                    break
            if found:
                break
        if s is None:
            s = ratio  # non-integer; keep as float

    info['inferred_stride'] = s

    # Solve for padding p using formula (assume stride s)
    k = info.get('kernel_size')
    s_try = info['inferred_stride']
    if (k is not None) and (s_try is not None) and isinstance(s_try, int):
        # invert formula: L_out = floor((L_in + 2p - (k-1) -1)/s + 1)
        # we can solve for p (choose p integer >=0) by brute force search small p
        found_p = None
        for p in range(0, 10):
            L_calc = math.floor((L_in + 2*p - (k-1) - 1) / s_try + 1)
            if L_calc == info['L_out']:
                found_p = p
                break
        info['inferred_padding'] = found_p
    else:
        info['inferred_padding'] = None

    return info

def inspect_down_block(model, block_name="enc.32_down", test_length=64):
    b = get_submodule_by_name(model, block_name)
    print(f"Inspecting block: {block_name}\n{'='*50}")
    # iterate over named submodules (works for scripted)
    # try to access attributes in order: norm0, conv0, norm1, conv1, skip
    for nm in ("norm0", "conv0", "norm1", "conv1", "skip"):
        if hasattr(b, nm) or (isinstance(getattr(b, "_modules", {}), dict) and nm in b._modules):
            try:
                sub = getattr(b, nm) if hasattr(b, nm) else b._modules[nm]
            except Exception:
                sub = b._modules[nm]
            print(f"\nSubmodule: {nm}")
            # print repr for quick view
            try:
                print(" repr:", sub)
            except Exception:
                pass
            # Weight info + probe
            info = infer_conv_attributes(sub, test_length=test_length)
            for k,v in info.items():
                print(f"  {k}: {v}")
        else:
            print(f"\nSubmodule: {nm} -> NOT FOUND")

In [12]:
inspect_down_block(model, "dec.16_up", test_length=64)


Inspecting block: dec.16_up

Submodule: norm0
 repr: RecursiveScriptModule(original_name=GroupNorm)
  weight_shape: (256,)
  has_bias: True
  probe_success: False

Submodule: conv0
 repr: RecursiveScriptModule(original_name=Conv1d)
  weight_shape: (256, 256, 3)
  kernel_size: 3
  out_channels: 256
  in_channels: 256
  has_bias: True
  probe_success: True
  L_in: 64
  L_out: 128
  inferred_stride: 0.5
  inferred_padding: None

Submodule: norm1
 repr: RecursiveScriptModule(original_name=GroupNorm)
  weight_shape: (256,)
  has_bias: True
  probe_success: False

Submodule: conv1
 repr: RecursiveScriptModule(original_name=Conv1d)
  weight_shape: (256, 256, 3)
  kernel_size: 3
  out_channels: 256
  in_channels: 256
  has_bias: True
  probe_success: True
  L_in: 64
  L_out: 64
  inferred_stride: 1
  inferred_padding: 1

Submodule: skip
 repr: RecursiveScriptModule(original_name=Conv1d)
  weight_shape: (256, 256, 1)
  kernel_size: 1
  out_channels: 256
  in_channels: 256
  has_bias: True
  pro

In [6]:
import torch
import math
import torch.nn.functional as F

def choose_group_norm_groups(num_channels: int, preferred: int = 32) -> int:
    max_g = min(preferred, num_channels)
    for g in range(max_g, 0, -1):
        if num_channels % g == 0:
            return g
    return 1

def safe_get_param_tensor(module, name):
    """Return parameter tensor for scripted or regular module, or None."""
    # Try attribute access
    try:
        param = getattr(module, name)
        if isinstance(param, torch.Tensor):
            return param
    except Exception:
        pass
    # Try _parameters dict
    try:
        pm = getattr(module, "_parameters", None)
        if isinstance(pm, dict) and name in pm and isinstance(pm[name], torch.Tensor):
            return pm[name]
    except Exception:
        pass
    # Try named_parameters iteration
    try:
        for n, p in module.named_parameters(recurse=False):
            if n == name:
                return p
    except Exception:
        pass
    return None

def safe_call_module(module, x):
    """Call scripted or regular module with input x, try .__call__ then .forward."""
    try:
        return module(x)
    except Exception:
        try:
            return module.forward(x)
        except Exception as e:
            raise RuntimeError(f"Module call failed: {e}")

def probe_conv(conv, test_length=16):
    info = {}
    # infer weight
    weight = safe_get_param_tensor(conv, "weight")
    bias = safe_get_param_tensor(conv, "bias")
    if weight is not None:
        ws = tuple(weight.shape)
        # conv weight shape is (out, in, kernel)
        if len(ws) >= 3:
            info["out_channels"] = int(ws[0])
            info["in_channels"] = int(ws[1])
            info["kernel_size"] = int(ws[2])
        info["weight_shape"] = ws
    else:
        info["weight_shape"] = None

    info["has_bias"] = bool(bias is not None)

    # do a forward pass probe if we know in_channels
    if "in_channels" in info:
        x = torch.randn(1, info["in_channels"], test_length)
        try:
            y = safe_call_module(conv, x)
            info["L_in"] = test_length
            info["L_out"] = int(y.shape[-1])
            # infer stride by equation heuristics
            L_in, L_out = info["L_in"], info["L_out"]
            if L_out == 0:
                info["inferred_stride"] = None
            elif L_in % L_out == 0:
                info["inferred_stride"] = int(L_in // L_out)
            else:
                # try candidates
                s = None
                k = info.get("kernel_size", None)
                for candidate in (1,2,3,4):
                    for p in range(0,5):
                        L_calc = math.floor((L_in + 2*p - ( (k-1) if k else 0 ) - 1) / candidate + 1)
                        if L_calc == L_out:
                            s = candidate
                            info["inferred_padding"] = p
                            break
                    if s is not None:
                        break
                info["inferred_stride"] = s
        except Exception as e:
            info["probe_error"] = str(e)
    return info

def inspect_decoder_block(model, block_name="dec.16_up", test_length=16, preferred_gn=32):
    # locate block (works for scripted & normal)
    parts = block_name.split(".")
    m = model
    for p in parts:
        if hasattr(m, p):
            m = getattr(m, p)
        elif isinstance(getattr(m, "_modules", None), dict) and p in m._modules:
            m = m._modules[p]
        else:
            raise KeyError(f"Cannot find submodule '{block_name}' (stuck at '{p}')")

    print(f"\n=== Inspecting block: {block_name} ===")
    for name, sub in getattr(m, "_modules", {}).items():
        print(f"\n-- submodule: {name} (type: {type(sub).__name__}) --")
        # GroupNorm inference
        if "norm" in name.lower():
            # try to read attributes first
            num_groups = None
            num_channels = None
            try:
                num_groups = getattr(sub, "num_groups")
            except Exception:
                pass
            # try to infer num_channels from weight param
            w = safe_get_param_tensor(sub, "weight")
            if w is not None and isinstance(w, torch.Tensor):
                num_channels = int(w.numel()) if w.dim() == 1 else int(w.shape[0])
            # fallback try buffers/named_parameters
            if num_channels is None:
                try:
                    params = list(sub.named_parameters(recurse=False))
                    for n,p in params:
                        if n == "weight":
                            num_channels = int(p.numel()) if p.dim()==1 else int(p.shape[0])
                except Exception:
                    pass

            if num_channels is not None and num_groups is None:
                num_groups = choose_group_norm_groups(num_channels, preferred=preferred_gn)

            print(f"  inferred num_channels: {num_channels}, num_groups: {num_groups}")

        # Conv inference
        if "conv" in name.lower() or "skip" == name.lower():
            try:
                info = probe_conv(sub, test_length=test_length)
                for k,v in info.items():
                    print(f"  {k}: {v}")
            except Exception as e:
                print(f"  conv probe failed: {e}")

    # Also print any top-level attrs that might exist (safely)
    print("\nTop-level attributes (safe list):")
    for attr in ("in_channels","out_channels","kernel_size","stride","padding"):
        try:
            val = getattr(m, attr)
            print(f"  {attr}: {val}")
        except Exception:
            pass

# Example usage:
# model = torch.load("model.pt", map_location="cpu")
# inspect_decoder_block(model, "dec.16_up", test_length=16)


In [7]:
inspect_block(model, "dec.16_up", test_length=16)



=== Inspecting dec.16_up ===

Submodule: norm0
 type: <class 'torch.jit._script.RecursiveScriptModule'>


AttributeError: 'RecursiveScriptModule' object has no attribute 'num_channels'

In [89]:
sub = getattr(model.enc, '64_block0', None)
norm = getattr(sub, 'norm0', None)

# print(sub.norm0.in_channels)


print(sub.conv1.in_channels)
print(sub.conv1.out_channels)
for name, module in sub.conv1.named_modules():
    print(name, module)
    print(module.weight.shape)
    print(module.bias.shape)




print(type(sub.conv1))

128
128
 RecursiveScriptModule(original_name=Conv1d)
torch.Size([128, 128, 3])
torch.Size([128])
<class 'torch.jit._script.RecursiveScriptModule'>


In [63]:
print(dir(sub.norm0))


['T_destination', '__annotations__', '__bool__', '__call__', '__class__', '__contains__', '__copy__', '__deepcopy__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__jit_unused_properties__', '__le__', '__len__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__reduce_package__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_apply', '_backward_hooks', '_backward_pre_hooks', '_buffers', '_c', '_call_impl', '_compiled_call_impl', '_concrete_type', '_constants_set', '_construct', '_disable_script_meta', '_finalize_scriptmodule', '_forward_hooks', '_forward_hooks_always_called', '_forward_hooks_with_kwargs', '_forward_pre_hooks', '_forward_pre_hooks_with_kwargs', '_get_backward_hooks', '_get_backward_pre_hooks', '_get_name', '_initial

TypeError: Module._named_members() missing 1 required positional argument: 'get_members_fn'

In [90]:
sub = None
i = 0
for block, m in model.enc.named_modules():   # or named_children() if you only want direct children
    # print(name)  # uncomment to inspect all names
    for name, sub_module in m.named_modules():
        original_name = sub_module.original_name if hasattr(sub_module, 'original_name') else name
        if original_name == 'GroupNorm':
            print(sub_module.num_groups)
        # if original_name == 'Conv1d':
        #     print(name)
        #     # print(f'Block {i} - found Conv1d:')
        #     i += 1
        #     print(sub_module.in_channels)
        #     print(sub_module.out_channels)
        #     # for name, module in sub_module.named_modules():
        #     #     print(name, module)
        #     #     print(module.weight.shape)
        #     #     print(module.bias.shape)


32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
32
